# Simulation analysis

After autorunning the simulation several times, with different input values,\
lets do a first simple analysis on values, errors and how reliable it really is.\
As these work as personal study and later presentation material, it is not expected\
for the plots and procedures to be fully explained here.

* More info and explaining of the plots is on ```testsimul.ipynb``` and the [SOTN-009 Technote](https://sotn-009.lsst.io})

* The ```.csv``` files referenced here ~surely~ won't be uploaded to the repo to avoid large storage sizes 

* Not auto-runnable notebok.

Auth: D. Hurtado / Suzakuuu

In [ ]:
from datetime import datetime, timedelta # For days_between function
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import os

from astropy.time import Time
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.dateTime import (getDayObsEndTime, getDayObsStartTime)
from lsst.summit.utils.efdUtils import (getEfdData, makeEfdClient)

import testsimul

# Create the client for InfluxQL
efd_client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=False)

# Autorun settings

In [ ]:
# Functions to fetch for the dataframes, copied from an old notebook
def timestamp_to_utc(timestamp):
    """
    Takes the timestamp in string value and
    converts it to a query readable timestamp
    
    Parameter
    ---------
    Timestamp: string (YYYY-MM-DD hh:mm:ss.ms)

    Returns:
    --------
    EFD-readable timestamp
    """
    timestamp = pd.to_datetime(str(timestamp))
    new_stamp = timestamp.value
    step = int(0.01 * 1e10)
    rounded_stamp = (new_stamp // step) * step
    timestamp = Time(pd.to_datetime(rounded_stamp), scale="utc")

    return timestamp


def days_between(start, end):
    """
    Creates a list of YYYYMMDD days for analysis
    
    Parameters:
    -----------
    start : int (YYYYMMDD)
        Start date
    end : int (YYYYMMDD)
        End date
    
    Returns:
    --------
    List with YYYYMMDD in between dates given
    """
    start_date = datetime.strptime(str(start), "%Y%m%d")
    end_date = datetime.strptime(str(end), "%Y%m%d")
    
    days = []
    current = start_date
    while current <= end_date:
        days.append(int(current.strftime("%Y%m%d")))
        current += timedelta(days=1)
    
    return days

def per_minute(df):
    """
    Take an input dataframe and groups it by minutes

    Parameter
    ---------
    df: dataframe with first column timestamp
        in format YYYY-MM-DD hh:mm:ss

    Returns:
    --------
    Grouped dataframe
    """
    out = df.copy()
    out['minute'] = out['timestamp'].dt.floor('min')         # 1 bin per minute
    num = out.select_dtypes('number').columns                # average numeric cols
    return out.groupby('minute')[num].mean()


In [ ]:
# Historic analysis config
# Arbitrary days
day_start = 20260508
day_end   = 20260510

daysinbetween = days_between(day_start, day_end)
print(f"We have {len(daysinbetween)} days to analyze")

start_time =  timestamp_to_utc(getDayObsStartTime(day_start))
end_time   =  timestamp_to_utc(getDayObsEndTime(day_end))

# Fetch dataframes
df_ringss = getEfdData(
    client  = efd_client,
    topic   = 'lsst.sal.ESS.logevent_ringssMeasurement',
    columns = ['fwhmFree', 'fwhmScintillation', 
                'fwhmSector', 'wind', 'flux'],
    begin   = start_time,
    end     = end_time
                        )

df_dimm= getEfdData(
    client  = efd_client,
    topic   = 'lsst.sal.DIMM.logevent_dimmMeasurement',
    columns = ['fwhm', 'r0', 'flux'],
    begin   = start_time,
    end     = end_time,
                )

# Reset EFD timestamp index into a normal column
df_ringss = df_ringss.reset_index()
df_ringss = df_ringss.rename(columns={'index':'timestamp'})
df_dimm = df_dimm.reset_index()
df_dimm = df_dimm.rename(columns={'index':'timestamp'})

# Group together by each minute both dataframes
ringss_min = per_minute(df_ringss)
dimm_min   = per_minute(df_dimm)

# Suffixes for shared names (e.g. 'flux' exists in both)
df_minutes = ringss_min.join(dimm_min, how='inner',
                          lsuffix='_ringss', rsuffix='_dimm')

print(f'Pachon RINGSS data points by minutes: {len(ringss_min)}')
print(f'Rubin DIMM data points by minutes: {len(dimm_min)}')
print(f'Coinciding minutes: {len(df_minutes)}')
df_minutes.head()

### Autorun

In [ ]:
# Run d_testsimul with the OBSERVED conditions (RINGSS sector seeing + wind)
# input the observed reference values onto that same row as obs_*
out = 'testsimul_results.csv'

max_runs = None # Change when running correctly
n = len(df_minutes) if max_runs is None else min(max_runs, len(df_minutes))

print(f'Running {n} simulations')
run = (n * 95 / 60) # Each run takes at most 95 seconds
if run > 90: print(f'Estimated {run/60:.1f} hours')
else: print(f'Estimated {run:.1f} minutes')
    
    
times = []
step = max(1, n // 10)  # print progress every ~10% of the runs
for i in range(n):
    ringss_sector = round(df_minutes['fwhmSector'].iloc[i], 3)
    wind          = round(df_minutes['wind'].iloc[i], 3)
    dimm_fwhm     = round(df_minutes['fwhm'].iloc[i], 3)
    seed0 = int(np.random.SeedSequence().generate_state(1, dtype=np.uint32)[0])

    # Run the simulation with observed sector seeing and wind
    t0 = time.time()
    d_testsimul.main(['--seeing', f'{ringss_sector}', '--wind', f'{wind}',
                      '--seed0', f'{seed0}', '--verbose', '', '--display', ''])  # verb/display off
    secs = round(time.time() - t0, 1)
    times.append(secs)

    # Append observed reference values to the row d_testsimul just wrote
    df_csv = pd.read_csv(out)
    obs_vals = {
        'obs_fwhmSector':        ringss_sector,
        'obs_fwhmFree':          df_minutes['fwhmFree'].iloc[i],
        'obs_fwhmScintillation': df_minutes['fwhmScintillation'].iloc[i],
        'obs_wind':              wind,
        'obs_fwhmDIMM':          dimm_fwhm,
                }
    for col, val in obs_vals.items():
        df_csv.loc[df_csv.index[-1], col] = val
    df_csv.to_csv(out, index=False)

    if i % step == 0 or i == n - 1:
        print(f'Sim {i}/{n} with Seeing: {ringss_sector}" and Wind: {wind} m/s in {secs}s')

if np.sum(times) > 90:
    print(f'\nDone! {n} runs appended to {out} in {np.sum(times) / 3600:.1f} hours.')
else: 
    print(f'\nDone! {n} runs appended to {out} in {np.sum(times) / 60:.1f} min.')

In [ ]:
# Run d_testsimul with tuples of Seeing and wind
# input reference values onto that same row as obs_*
'''
out = 'windsee_results.csv'

s = []
w = []
for i in np.arange(0.3, 1.9, 0.2): s.append(i)
for i in np.arange(0, 22, 3): w.append(i)

n = len(s) * len(w)

print(f'Running {n} simulations')
run = (n * 95 / 60) # Each run takes at most 95 seconds
if run > 90: print(f'Estimated {run/60:.1f} hours')
else: print(f'Estimated {run:.1f} minutes')

i = 0

times = []
#step = max(1, n // 10)  # print progress every ~10% of the runs
for see in s:
    for win in w: #There has to be a more elegant way of doing this
        i += 1
        seed0 = int(np.random.SeedSequence().generate_state(1, dtype=np.uint32)[0])
        
        t0 = time.time()
        testsimul.main(['--seeing', f'{round(see,2)}', '--wind', f'{round(win,2)}',
                          '--seed0', f'{seed0}', '--verbose', '', '--display', ''])  # verb/display off
        secs = round(time.time() - t0, 1)
        times.append(secs)
        
        df_csv = pd.read_csv(out)
        df_csv.to_csv(out, index=False)
    
    print(f'Sim {i}/{n} with Seeing: {see}" and Wind: {win} m/s in {secs}s')

if np.sum(times) > 90:
    print(f'\nDone! {n} runs appended to {out} in {np.sum(times) / 3600:.1f} hours.')
else: 
    print(f'\nDone! {n} runs appended to {out} in {np.sum(times) / 60:.1f} min.')

'''

# Analysis

In [ ]:
df_results = pd.read_csv('testsimul_results.csv')
print(f'Python runs: {len(df_results)} runs, {len(df_results.columns)} columns')

# df_winsee = pd.read_csv('windsee_results.csv')
# print(f'From Wind x Seeing: {len(df_windsee)} runs, {len(df_windsee.columns)} columns')

In [ ]:
df_pro = pd.read_csv('idl_testsimul_results.csv')
print(f'IDL runs: {len(df_pro)} runs, {len(df_pro.columns)} columns')

# df_winsee = pd.read_csv('windsee_results.csv')
# print(f'From Wind x Seeing: {len(df_windsee)} runs, {len(df_windsee.columns)} columns')

In [ ]:
# Drop outliers
max_see = 4    # arcsec
#min_wind = 4  # arcsec
df_results = df_results[df_results['see_arcsec'] < max_see]
#df_results = df_results[df_results['obs_wind'] > min_wind]
print(f'Dropped outliers, {len(df_results)} rows remain')

## Historical

### Seeing vs sample

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(10, 7))


plt.plot(df_results['see_arcsec'], color='xkcd:azul', alpha=0.8, label='Scintillation')
         
plt.plot(df_results['see2_arcsec'], color='xkcd:teal', alpha=0.8, label='Sector')

plt.plot(df_results['fsee_arcsec'], color='xkcd:dried blood', alpha=0.8, label='Free atmosphere')

plt.plot(df_results['ron_el'], color='xkcd:mustard', alpha=0.8, label='Read Out Noise (e-)')


plt.xlabel('Sample')
plt.ylabel('Measured seeing"')
plt.title('Seeing through samples')
plt.ylim(0.2,4)
#plt.xlim(0,1400)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(10, 7))


plt.plot(df_pro['see_arcsec'], color='xkcd:wine red', alpha=0.8, label='IDL Scintillation')
         
plt.plot(df_pro['see2_arcsec'], color='xkcd:vermillion', alpha=0.8, label='IDL Sector')

plt.plot(df_pro['fsee_arcsec'], color='xkcd:fire engine red', alpha=0.8, label='IDL Free atmosphere')


plt.xlabel('Sample')
plt.ylabel('Measured seeing"')
plt.title('Seeing through samples')
plt.legend()
plt.grid(True)
plt.show()

### Sample Z vs Scale


In [ ]:
fig, ax = plt.subplots(8, 1, figsize=(8, 5), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0)

ax[0].set_title('Turbulence profile over layers')
ax[0].plot(df_results[f'J_1e13_{7}'], color='xkcd:denim')
ax[0].set_ylabel(df_results[f'Z_km_{7}'][1])
ax[0].grid(True)
#ax[0].fill_between(df_results[f'J_1e13_{7}'], color='xckd:denim', alpha=0.5)

ax[1].plot(df_results[f'J_1e13_{6}'], color='xkcd:dark sea green')
ax[1].set_ylabel(df_results[f'Z_km_{6}'][1])
ax[1].grid(True)
#ax[1].fill_between(df_results[f'J_1e13_{6}'], color='xckd:dark sea green', alpha=0.5)

ax[2].plot(df_results[f'J_1e13_{5}'], color='xkcd:tree green')
ax[2].set_ylabel(df_results[f'Z_km_{5}'][1])
ax[2].grid(True)
#ax[2].fill_between(df_results[f'J_1e13_{5}'], color='xckd:tree green', alpha=0.5)

ax[3].plot(df_results[f'J_1e13_{4}'], color='xkcd:electric green')
ax[3].set_ylabel(df_results[f'Z_km_{4}'][1])
ax[3].grid(True)
#ax[3].fill_between(df_results[f'J_1e13_{4}'], color='xckd:electric green', alpha=0.5)

ax[4].plot(df_results[f'J_1e13_{3}'], color='xkcd:apple green')
ax[4].set_ylabel(df_results[f'Z_km_{3}'][1])
ax[4].grid(True)
#ax[4].fill_between(df_results[f'J_1e13_{3}'], color='xckd:apple green', alpha=0.5)

ax[5].plot(df_results[f'J_1e13_{2}'], color='xkcd:bluish green')
ax[5].set_ylabel(df_results[f'Z_km_{2}'][1])
ax[5].grid(True)
#ax[5].fill_between(df_results[f'J_1e13_{2}'], color='xckd:bluish green', alpha=0.5)

ax[6].plot(df_results[f'J_1e13_{1}'], color='xkcd:dried blood')
ax[6].set_ylabel(df_results[f'Z_km_{1}'][1])
ax[6].grid(True)
#ax[6].fill_between(df_results[f'J_1e13_{1}'], color='xckd:dried blood', alpha=0.5)

ax[7].plot(df_results[f'J_1e13_{0}'], color='xkcd:red')
ax[7].set_ylabel(df_results[f'Z_km_{0}'][1])
ax[7].grid(True)
#ax[7].fill_between(df_results[f'J_1e13_{0}'], color='xckd:red', alpha=0.5

plt.xlabel('Simulation number')
plt.ylim(-0.1,5)
plt.xlim(600,1400)
plt.yticks([])
plt.show()

### Sample Z vs Scale (IDL)

In [ ]:
fig, ax = plt.subplots(8, 1, figsize=(8, 5), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0)

ax[0].set_title('Turbulence profile over layers (IDL)')
ax[0].plot(df_pro[f'J_1e13_{7}'], color='xkcd:denim')
ax[0].set_ylabel(df_pro[f'Z_km_{7}'][1])
ax[0].grid(True)

ax[1].plot(df_pro[f'J_1e13_{6}'], color='xkcd:dark sea green')
ax[1].set_ylabel(df_pro[f'Z_km_{6}'][1])
ax[1].grid(True)

ax[2].plot(df_pro[f'J_1e13_{5}'], color='xkcd:tree green')
ax[2].set_ylabel(df_pro[f'Z_km_{5}'][1])
ax[2].grid(True)

ax[3].plot(df_pro[f'J_1e13_{4}'], color='xkcd:electric green')
ax[3].set_ylabel(df_pro[f'Z_km_{4}'][1])
ax[3].grid(True)

ax[4].plot(df_pro[f'J_1e13_{3}'], color='xkcd:apple green')
ax[4].set_ylabel(df_pro[f'Z_km_{3}'][1])
ax[4].grid(True)

ax[5].plot(df_pro[f'J_1e13_{2}'], color='xkcd:bluish green')
ax[5].set_ylabel(df_pro[f'Z_km_{2}'][1])
ax[5].grid(True)

ax[6].plot(df_pro[f'J_1e13_{1}'], color='xkcd:dried blood')
ax[6].set_ylabel(df_pro[f'Z_km_{1}'][1])
ax[6].grid(True)

ax[7].plot(df_pro[f'J_1e13_{0}'], color='xkcd:red')
ax[7].set_ylabel(df_pro[f'Z_km_{0}'][1])
ax[7].grid(True)

plt.xlabel('Sample')
plt.ylim(-0.1, 5)
plt.xlim(0, len(df_pro))
plt.yticks([])
plt.show()

### Seeing vs Seeing

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))


plt.plot(df_results['input_seeing_arcsec'], df_results['see2_arcsec'], 
         's', color='xkcd:teal', alpha=0.7, label='Python Input vs Sector')
         
plt.plot(df_pro['input_seeing_arcsec'], df_pro['see2_arcsec'], 
         '.', color='xkcd:dried blood', alpha=0.7, label='IDL Input vs Sector')



lims = [df_results['input_seeing_arcsec'].min(), int(df_results['input_seeing_arcsec'].min()) + 3]
plt.plot(lims, lims, 'k--', label='1:1 (perfect)')

plt.xlabel('Input seeing"')
plt.ylabel('Measured seeing"')
plt.title('Python vs IDL')
plt.ylim(0,3)
plt.xlim(0,3)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Logarithmic curve fit: see2 (sector) vs input seeing.
# Model: y = a*ln(x) + b
x = df_results['input_seeing_arcsec'].to_numpy(dtype=float)
y = df_results['see2_arcsec'].to_numpy(dtype=float)

# keep finite, positive x (ln needs x > 0)
ok = np.isfinite(x) & np.isfinite(y) & (x > 0)
x, y = x[ok], y[ok]

# fit y = a*ln(x) + b  ->  linear fit of y vs ln(x)
a, b = np.polyfit(np.log(x), y, 1)

# R^2
yfit   = a * np.log(x) + b
ss_res = np.sum((y - yfit) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f'Sector = {a:.3f} * ln(input) + {b:.3f}   (R^2 = {r2:.3f}, N = {len(x)})')

# smooth curve for plotting
xs = np.linspace(x.min(), x.max(), 200)
ys = a * np.log(xs) + b

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))


plt.plot(df_results['obs_fwhmScintillation'], df_results['see_arcsec'], 
         'x', color='xkcd:teal', alpha=0.8, label='Pachon Scint vs Scint')
         
plt.plot(df_results['input_seeing_arcsec'], df_results['see2_arcsec'], 
         's', color='xkcd:cobalt blue', alpha=0.7, label='Pachon Sector vs Sector*')

         
plt.plot(df_results['input_seeing_arcsec'], df_results['fsee_arcsec'], 
         '.', color='xkcd:azul', alpha=0.7, label='Pachon Sector vs Free')


lims = [df_results['input_seeing_arcsec'].min(), int(df_results['input_seeing_arcsec'].min()) + 3]
plt.plot(lims, lims, 'k--', label='1:1 (perfect)')


plt.xlabel('Input seeing"')
plt.ylabel('Measured seeing"')
plt.title('Sim vs Input (Pachon) seeing')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))


plt.plot(df_pro['input_seeing_arcsec'], df_pro['see_arcsec'], 
         '.', color='xkcd:blue', alpha=1, label='IDL Input vs Scint')
         
plt.plot(df_pro['input_seeing_arcsec'], df_pro['see2_arcsec'], 
         'x', color='xkcd:mustard', alpha=1, label='IDL Input vs Sector')

plt.plot(df_pro['input_seeing_arcsec'], df_pro['fsee_arcsec'], 
         'o', color='xkcd:vermillion', alpha=0.3, label='IDL Input vs Free')


plt.xlabel('Input seeing"')
plt.ylabel('Measured seeing"')
plt.title('IDL Sim Input vs Calculated')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))

plt.plot(df_results['input_seeing_arcsec'], df_results['obs_fwhmDIMM'], 
         '.', color='xkcd:red', alpha=1, label='Pachon Sector vs Rubin Sector')


plt.xlabel('RINGSS seeing"')
plt.ylabel('DIMM seeing"')
plt.title('RINGSS vs DIMM (not simulation)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))
         
plt.plot(df_results['see_arcsec'], df_results['see2_arcsec'], 
         '.', color='xkcd:dried blood', alpha=0.8, label='Sim Scint vs Sim Sector')

plt.plot(df_results['obs_fwhmScintillation'], df_results['obs_fwhmSector'], 
         '.', color='xkcd:vermillion', alpha=0.8, label='Pachon Scint vs Pachon Sector')


lims = [df_results['see_arcsec'].min(), df_results['see2_arcsec'].max()]
plt.plot(lims, lims, 'k--', label='1:1 (perfect)')


plt.xlabel('Scint seeing"')
plt.ylabel('Sector seeing"')
plt.title('Scintillation vs Sector seeing')
plt.legend()
plt.grid(True)
plt.show()

### Wind vs Wind

In [ ]:
# Measured seeing vs. input seeing
plt.figure(figsize=(7, 6))
plt.plot(df_results['obs_wind'], df_results['wind'], 'o', color='xkcd:orange red', label='Sim vs Input')
#plt.plot(df_results['input_wind_ms'], df_results['obs_wind'],  'x', label='1:1')

lims = [df_results['obs_wind'].min(), df_results['obs_wind'].max()]
plt.plot(lims, lims, 'k--', label='1:1 (perfect)')

plt.xlabel('Input Wind m/s')
plt.ylabel('Measured Wind m/s')
plt.title('Sim vs Input Wind speed')
plt.legend()
plt.grid(True)
plt.show()
plt.clf()

### Wind vs Seeing

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(df_results['input_wind_ms'], df_results['input_seeing_arcsec'], 'o', color='xkcd:dried blood')
ax1.set_xlabel('Input Wind m/s')
ax1.set_ylabel('Input seeing"')
ax1.set_title('Input Wind vs Seeing')
ax1.grid(True)

ax2.plot(df_results['input_wind_ms'], df_results['see_arcsec'], 'o', color='xkcd:teal')
ax2.set_xlabel('Input Wind m/s')
ax2.set_ylabel('Derived seeing"')
ax2.set_title('Simulated Wind vs Seeing')
ax2.grid(True)


plt.show()

### Angular power spectrum

In [ ]:
# Angular power spectrum (same plot as d_statmom / d_testsimul)
m = np.arange(21)                                   # angular frequency axis
power_cols = [f'power_m_{j}' for j in m]

# mean and std of each power_m across the repeated runs at each input seeing
grp  = df_results.groupby('input_seeing_arcsec')[power_cols]
mean = grp.mean()
std  = grp.std().fillna(0.0)                        # 0 if a seeing has a single run

# pick a few seeing values spanning the range
seeing = sorted(df_results['input_seeing_arcsec'].unique())
selected = [seeing[-1], seeing[0], seeing[2]]

plt.figure(figsize=(7, 5))
for s in selected:
    y   = mean.loc[s].to_numpy(dtype=float)
    err = std.loc[s].to_numpy(dtype=float)
    plt.errorbar(m, np.maximum(y, 1e-12), yerr=err, marker='o', capsize=3,
                 label=f'seeing {s}\"')

# shared noise floor (mean anoise over all runs)
plt.axhline(max(float(df_results['anoise'].mean()), 1e-12), ls=':', color='b', label='noise')
plt.yscale('log')
plt.xlim(0, 20)
plt.xticks(np.arange(0, 21, 1))
plt.xlabel('m')
plt.ylabel('Power')
plt.title('Angular power spectrum vs input seeing (mean \u00b1 std of repeats)')
plt.legend()
plt.grid(True)
plt.show()


## IDL Plots port

In [ ]:
# Angular power spectrum (IDL) -- ports statmom.pro / testsimul.pro: pow, cov, noise vs m
m = np.arange(21)
pw = df_pro[[f'power_m_{j}' for j in m]].mean().to_numpy()
cv = df_pro[[f'cov_m_{j}' for j in m]].mean().to_numpy()
an = float(df_pro['anoise'].mean())

plt.figure(figsize=(7, 5))
plt.semilogy(m, np.maximum(pw, 1e-12), 'k-o', label='Power (var)')
plt.semilogy(m, np.maximum(cv, 1e-12), 'r--', label='Covariance (cov)')
plt.axhline(max(an, 1e-12), ls=':', color='b', label='Noise (anoise)')
plt.xlabel('m'); plt.ylabel('Power')
plt.title('IDL angular power spectrum (mean of runs)')
plt.xlim(0, 20); plt.xticks(np.arange(0, 21, 1))
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Mode correlation rho = cov/pow (IDL) -- ports xd5.pro
m = np.arange(21)
pw = df_pro[[f'power_m_{j}' for j in m]].mean().to_numpy()
cv = df_pro[[f'cov_m_{j}' for j in m]].mean().to_numpy()
rho = np.divide(cv, pw, out=np.zeros_like(cv), where=pw > 0)

plt.figure(figsize=(7, 5))
plt.plot(m, rho, 'k-o')
plt.xlabel('m'); plt.ylabel('Correlation  cov/pow')
plt.title('IDL mode correlation (mean of runs)')
plt.xlim(0, 20); plt.xticks(np.arange(0, 21, 1))
plt.grid(True); plt.show()

In [ ]:
# Seeing per sample (IDL) -- ports xd5.pro: scint (see), sector (see2), free-atm (fsee)
x = np.arange(len(df_pro))
plt.figure(figsize=(12, 5))
plt.plot(x, df_pro['see_arcsec'],  '-',  color='xkcd:blue',    label='Scintillation (see)')
plt.plot(x, df_pro['see2_arcsec'], '--', color='xkcd:magenta', label='Sector (see2)')
plt.plot(x, df_pro['fsee_arcsec'], ':',  color='xkcd:green',   label='Free-atm (fsee)')
plt.xlabel('Sample'); plt.ylabel('Seeing [arcsec]')
plt.title('IDL seeing per sample')
plt.legend(); plt.grid(True); plt.show()

## IDL vs Python comparison

In [ ]:
# Common clean subset for a fair IDL-vs-Python comparison
lo, hi = 0.3, 3.0
py_c  = df_results[(df_results.input_seeing_arcsec >= lo) & (df_results.input_seeing_arcsec <= hi)]
idl_c = df_pro[(df_pro.input_seeing_arcsec >= lo) & (df_pro.input_seeing_arcsec <= hi)]
print(f'Python clean: {len(py_c)} runs    IDL clean: {len(idl_c)} runs')

In [ ]:
# Angular power spectrum: IDL vs Python (mean power per mode)
m = np.arange(21)
pcols = [f'power_m_{j}' for j in m]
pw_py  = py_c[pcols].mean().to_numpy()
pw_idl = idl_c[pcols].mean().to_numpy()

plt.figure(figsize=(7, 5))
plt.semilogy(m, np.maximum(pw_py, 1e-12),  'k-o', label='Python')
plt.semilogy(m, np.maximum(pw_idl, 1e-12), 'r-s', label='IDL')
plt.xlabel('m'); plt.ylabel('Power (var)')
plt.title('Angular power spectrum: IDL vs Python')
plt.xlim(0, 20); plt.xticks(np.arange(0, 21, 1))
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Measured vs input seeing: IDL vs Python (sector see2 and scint see)
plt.figure(figsize=(7, 6))
plt.plot(py_c.input_seeing_arcsec,  py_c.see2_arcsec,  '.', color='xkcd:blue',     alpha=0.4, label='Python sector (see2)')
plt.plot(idl_c.input_seeing_arcsec, idl_c.see2_arcsec, '.', color='xkcd:red',      alpha=0.4, label='IDL sector (see2)')
plt.plot(py_c.input_seeing_arcsec,  py_c.see_arcsec,   'x', color='xkcd:sky blue', alpha=0.3, label='Python scint (see)')
plt.plot(idl_c.input_seeing_arcsec, idl_c.see_arcsec,  'x', color='xkcd:orange',   alpha=0.3, label='IDL scint (see)')
lims = [lo, hi]
plt.plot(lims, lims, 'k--', label='1:1')
plt.xlabel('Input seeing "'); plt.ylabel('Measured seeing "')
plt.title('Seeing vs input: IDL vs Python')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Mean turbulence profile J(z) per layer: IDL vs Python
z = [df_pro[f'Z_km_{i}'][1] for i in range(8)]
jpy  = [py_c[f'J_1e13_{i}'].mean()  for i in range(8)]
jidl = [idl_c[f'J_1e13_{i}'].mean() for i in range(8)]
x = np.arange(8); w = 0.4

plt.figure(figsize=(9, 5))
plt.bar(x - w/2, jpy,  w, color='xkcd:denim',     label='Python')
plt.bar(x + w/2, jidl, w, color='xkcd:brick red', label='IDL')
plt.xticks(x, [f'{zz:g}' for zz in z])
plt.xlabel('Layer altitude [km]'); plt.ylabel('Mean J [1e-13 m$^{1/3}$]')
plt.title('Mean turbulence profile per layer: IDL vs Python')
plt.legend(); plt.grid(True, axis='y'); plt.show()

In [ ]:
# Wind distribution: IDL vs Python (normalized -- density=True so the ~2.5k Python
# samples and the ~290 IDL samples are compared as fractions, not raw counts)
bins = np.arange(0, 40, 2)
plt.figure(figsize=(7, 5))
plt.hist(py_c.wind,  bins=bins, density=True, alpha=0.5, color='xkcd:denim',     label=f'Python (n={len(py_c)})')
plt.hist(idl_c.wind, bins=bins, density=True, alpha=0.5, color='xkcd:brick red', label=f'IDL (n={len(idl_c)})')
plt.xlabel('Measured wind [m/s]'); plt.ylabel('Density')
plt.title('Wind distribution: IDL vs Python (normalized)')
plt.legend(); plt.grid(True, axis='y'); plt.show()

## Read-out noise impact (Python)

In [ ]:
# Read-out noise: RON 0 & 1 e- vs RON = 3.5 e- (Python)
m = np.arange(21)
pcols = [f'power_m_{j}' for j in m]
g01 = py_c[py_c.ron_el <= 1]                          # RON 0 and 1 e-
g35 = py_c[(py_c.ron_el >= 3) & (py_c.ron_el <= 4)]   # RON ~ 3.5 e-

plt.figure(figsize=(7, 7))
plt.semilogy(m, np.maximum(g01[pcols].mean(), 1e-12), 'o-', color='xkcd:cobalt blue', alpha=0.75, label=f'power, RON 0 & 1 e- (n={len(g01)})')
plt.semilogy(m, np.maximum(g35[pcols].mean(), 1e-12), 's-', color='xkcd:dried blood', alpha=0.75, label=f'power, RON = 3.5 e- (n={len(g35)})')
plt.xlabel('m'); plt.ylabel('Power')
plt.title('Read-out noise impact on the angular spectrum (Python)')
plt.xlim(0, 20); plt.xticks(np.arange(0, 21, 1))
plt.legend(); plt.grid(True); plt.show()

## Characteristics & quirks

In [ ]:
# Sim vs on-sky (Pachon RINGSS): per-channel validation (the real test)
d = df_results[(df_results.input_seeing_arcsec >= 0.3) & (df_results.input_seeing_arcsec <= 3.0)]
pairs = [('see2_arcsec', 'obs_fwhmSector',        'Sector',          'xkcd:red'),
         ('see_arcsec',  'obs_fwhmScintillation', 'Scintillation',   'xkcd:blue'),
         ('fsee_arcsec', 'obs_fwhmFree',          'Free-atmosphere', 'xkcd:green')]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for k, (sim, obs, name, c) in enumerate(pairs):
    mm = np.isfinite(d[sim]) & np.isfinite(d[obs])
    x, y = d.loc[mm, obs], d.loc[mm, sim]
    ax[k].plot(x, y, '.', color=c, alpha=0.3)
    lims = [0, max(x.max(), y.max()) * 1.05]
    ax[k].plot(lims, lims, 'k--', label='1:1')
    ax[k].set_title(f'{name}\nR={x.corr(y):.3f}, mean sim/obs={(y/x).mean():.2f}')
    ax[k].set_xlabel('Pachon on-sky "'); ax[k].set_ylabel('Simulation "')
    ax[k].grid(True); ax[k].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Channel offset quirk: scint & free-atm seeing relative to the sector (DIMM-like) seeing
d = df_results[(df_results.input_seeing_arcsec >= 0.3) & (df_results.input_seeing_arcsec <= 3.0)]
r_scint = (d.see_arcsec  / d.see2_arcsec).dropna()
r_free  = (d.fsee_arcsec / d.see2_arcsec).dropna()
plt.figure(figsize=(7, 5))
bins = np.arange(0, 1.4, 0.03)
plt.hist(r_scint, bins=bins, alpha=0.5, color='xkcd:blue',  label=f'see/see2  (mean {r_scint.mean():.2f})')
plt.hist(r_free,  bins=bins, alpha=0.5, color='xkcd:green', label=f'fsee/see2 (mean {r_free.mean():.2f})')
plt.axvline(1.0, ls='--', color='k', label='1:1 (= sector)')
plt.xlabel('ratio to sector seeing (see2)'); plt.ylabel('Count')
plt.title('Scintillation & free-atm sit below the sector seeing (stable offset)')
plt.legend(); plt.grid(True, axis='y'); plt.show()

In [ ]:
# Scintillation saturation: totvar (scint. index) and its correction factor vs input seeing
d = df_results[(df_results.input_seeing_arcsec >= 0.3) & (df_results.input_seeing_arcsec <= 3.0)]
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(d.input_seeing_arcsec, d.totvar, '.', color='xkcd:purple', alpha=0.3)
ax1.set_xlabel('Input seeing "'); ax1.set_ylabel('totvar (scintillation index)', color='xkcd:purple')
ax1.tick_params(axis='y', labelcolor='xkcd:purple')
ax2 = ax1.twinx()
ax2.plot(d.input_seeing_arcsec, 1 / (1 - 0.4 * d.totvar), '.', color='xkcd:orange', alpha=0.3)
ax2.set_ylabel('saturation correction  1/(1 - 0.4 totvar)', color='xkcd:orange')
ax2.tick_params(axis='y', labelcolor='xkcd:orange')
ax1.set_title('Scintillation saturation grows with seeing')
ax1.grid(True); plt.show()

In [ ]:
# Restoration quality: the NNLS fit residual (erms) grows with seeing
d = df_results[(df_results.input_seeing_arcsec >= 0.3) & (df_results.input_seeing_arcsec <= 3.0)]
plt.figure(figsize=(8, 5))
plt.plot(d.input_seeing_arcsec, d.erms, '.', color='xkcd:teal', alpha=0.25)
bins = np.arange(0.3, 3.1, 0.2); cen = 0.5 * (bins[:-1] + bins[1:])
bm = [d.erms[(d.input_seeing_arcsec >= bins[i]) & (d.input_seeing_arcsec < bins[i+1])].mean() for i in range(len(bins)-1)]
plt.plot(cen, bm, '-o', color='xkcd:dark red', label='binned mean')
plt.xlabel('Input seeing "'); plt.ylabel('NNLS fit residual  erms')
plt.title('Profile-restoration residual grows with seeing')
plt.legend(); plt.grid(True); plt.show()